# 词性标注实践：基于规则的方法（Brill Tagger）


基于规则的词性标注方法是最早应用于词性标注任务的一类方法。本案例重点实现书中介绍的
**基于转换的 Brill Tagger**——一种利用**错误驱动方法**学习转换规则的词性标注算法。

其核心步骤为：

1. **初始化**：对词典中的词，赋予其最常用的词性作为初始标注；对词典外的词，用简单规则赋值（如首字母大写 → 专有名词）。
2. **规则转换**：根据"补丁规则"（Patch Rule）对初始标注进行修正，补丁规则包含三类：
   - (a) 若某词词性为 $a$，且其所在上下文为 $C$，则转换为 $b$；
   - (b) 若某词词性为 $a$，且其具有词汇属性 $P$，则转换为 $b$；
   - (c) 若某词词性为 $a$，且其周边范围 $R$ 内有一个词具有词汇属性 $P$，则转换为 $b$。
3. **规则学习**：采用**错误驱动的有监督模板学习**——用现有词典和补丁模板分析训练语料，
   把错误汇总为三元组 $\langle tag_a, tag_b, n\rangle$，再用一组模板（如"前一个词标注为 $z$"）
   生成候选规则，选择**净修正数**（修复的错误数 − 引入的新错误数）最大的规则加入规则表，
   反复迭代直至无法再改进。

书中给出的经典例子是补丁规则 **"NN VB PREV-TAG TO"**：若一个词被标注为 NN，且前一个词标注为 TO（不定式 to），
则将其改为 VB——用于解决类似 **"to book a hotel"** 中单词 *book* 默认被误标为名词的问题。
本 Notebook 将从零构造一个小型训练语料，完整实现"初始化 → 规则学习 → 规则应用"的流程，
并复现这个例子。


## 1. 准备训练语料

构造一个小型的、已完成词性标注的英文训练语料（PTB 风格标签）。
刻意让 *book* 一词在语料中同时出现名词（NN）和动词（VB）两种用法，以制造歧义，供后续规则学习使用。

In [1]:
train_corpus = [
    [("I", "PRP"), ("read", "VBP"), ("a", "DT"), ("book", "NN"), (".", ".")],
    [("The", "DT"), ("book", "NN"), ("is", "VBZ"), ("good", "JJ"), (".", ".")],
    [("This", "DT"), ("book", "NN"), ("was", "VBD"), ("great", "JJ"), (".", ".")],
    [("My", "PRP$"), ("favorite", "JJ"), ("book", "NN"), ("is", "VBZ"), ("here", "RB"), (".", ".")],
    [("I", "PRP"), ("want", "VBP"), ("to", "TO"), ("book", "VB"), ("a", "DT"), ("flight", "NN"), (".", ".")],
    [("She", "PRP"), ("will", "MD"), ("book", "VB"), ("the", "DT"), ("room", "NN"), (".", ".")],
    [("They", "PRP"), ("plan", "VBP"), ("to", "TO"), ("book", "VB"), ("tickets", "NNS"), (".", ".")],
]

for i, sent in enumerate(train_corpus, 1):
    print(f"句{i}: " + " ".join(f"{w}/{t}" for w, t in sent))


句1: I/PRP read/VBP a/DT book/NN ./.
句2: The/DT book/NN is/VBZ good/JJ ./.
句3: This/DT book/NN was/VBD great/JJ ./.
句4: My/PRP$ favorite/JJ book/NN is/VBZ here/RB ./.
句5: I/PRP want/VBP to/TO book/VB a/DT flight/NN ./.
句6: She/PRP will/MD book/VB the/DT room/NN ./.
句7: They/PRP plan/VBP to/TO book/VB tickets/NNS ./.


## 2. 初始化：构建词典并赋予初始词性

对应原文步骤 (1)：*"对于词典中包含的词语，根据词语最常使用的词性设置初始值；
对于词典中没有的单词根据词性分析结果设置初始值（例如：以大写字母开头的设置为专有名词）。"*


In [2]:
from collections import Counter, defaultdict

def build_lexicon(corpus):
    """统计每个词在语料中最常见的词性，构建初始化词典"""
    counts = defaultdict(Counter)
    for sent in corpus:
        for w, t in sent:
            counts[w][t] += 1
    return {w: c.most_common(1)[0][0] for w, c in counts.items()}

lexicon = build_lexicon(train_corpus)

print(f"{'词':<10}{'各词性出现次数':<25}{'初始（最常用）词性'}")
for w in ["book", "to", "will", "a", "is"]:
    counts = Counter(t for sent in train_corpus for ww, t in sent if ww == w)
    print(f"{w:<10}{str(dict(counts)):<25}{lexicon[w]}")


词         各词性出现次数                  初始（最常用）词性
book      {'NN': 4, 'VB': 3}       NN
to        {'TO': 2}                TO
will      {'MD': 1}                MD
a         {'DT': 2}                DT
is        {'VBZ': 2}               VBZ


可以看到，**"book"** 在语料中被标注为 NN 的次数（4次）多于 VB（3次），
因此词典赋予它的初始（默认）词性是 **NN**——这会导致 "to book a flight" 这类句子中的
"book" 被初始标注器错误地打成名词，正是书中提到的默认标注错误问题。


In [3]:
def initial_tag_word(word, lexicon):
    """初始化打标：词典内查表；词典外按大写规则猜测为专有名词，否则默认 NN"""
    if word in lexicon:
        return lexicon[word]
    return "NNP" if word[0].isupper() else "NN"


def tag_corpus_baseline(corpus, lexicon):
    return [[initial_tag_word(w, lexicon) for w, _ in sent] for sent in corpus]


baseline_tags = tag_corpus_baseline(train_corpus, lexicon)

print("=== 仅用词典初始化后的标注结果（与标准答案对比） ===")
for sent, pred in zip(train_corpus, baseline_tags):
    words = [w for w, _ in sent]
    gold = [t for _, t in sent]
    marks = ["" if p == g else " <-- 错误" for p, g in zip(pred, gold)]
    line = " ".join(f"{w}/{p}{m}" for w, p, m in zip(words, pred, marks))
    print(line)


=== 仅用词典初始化后的标注结果（与标准答案对比） ===
I/PRP read/VBP a/DT book/NN ./.
The/DT book/NN is/VBZ good/JJ ./.
This/DT book/NN was/VBD great/JJ ./.
My/PRP$ favorite/JJ book/NN is/VBZ here/RB ./.
I/PRP want/VBP to/TO book/NN <-- 错误 a/DT flight/NN ./.
She/PRP will/MD book/NN <-- 错误 the/DT room/NN ./.
They/PRP plan/VBP to/TO book/NN <-- 错误 tickets/NNS ./.


## 3. 定义补丁规则模板

原文列举了多种模板，例如：

- 前一个（或者后一个）单词被标注为了 $z$
- 前面第二个（或者后面第二个）单词被标注为了 $z$
- 当前单词是（不是）首字母大写
- 前一个单词是（不是）首字母大写

本案例实现其中有代表性的 5 种模板，足以复现书中的核心例子。


In [4]:
templates = {
    "PREV-TAG":  lambda words, tags, i: tags[i - 1] if i - 1 >= 0 else None,
    "NEXT-TAG":  lambda words, tags, i: tags[i + 1] if i + 1 < len(tags) else None,
    "PREV2-TAG": lambda words, tags, i: tags[i - 2] if i - 2 >= 0 else None,
    "CURR-CAP":  lambda words, tags, i: words[i][0].isupper(),
    "PREV-CAP":  lambda words, tags, i: words[i - 1][0].isupper() if i - 1 >= 0 else None,
}
print("已定义的补丁规则模板:", list(templates.keys()))


已定义的补丁规则模板: ['PREV-TAG', 'NEXT-TAG', 'PREV2-TAG', 'CURR-CAP', 'PREV-CAP']


## 4. 基于错误驱动方法学习补丁规则

对应原文步骤 (2)：根据现有词典和补丁模板对训练语料进行分析，
把错误标注汇总为 $\langle tag_a, tag_b, n\rangle$ 形式的三元组，
再利用模板生成候选规则，选择"能修复的错误数 − 引入的新错误数"最大（即净修正数最大）的
规则加入规则列表，重复此过程直至无法再改进。


In [5]:
def learn_rules(corpus, lexicon, max_rules=10, verbose=True):
    sentences_words = [[w for w, _ in sent] for sent in corpus]
    gold_tags = [[t for _, t in sent] for sent in corpus]
    cur_tags = [[initial_tag_word(w, lexicon) for w in words] for words in sentences_words]

    learned_rules = []
    for round_idx in range(max_rules):
        # 4.1 找出当前所有标注错误的位置
        errors = []
        for s in range(len(corpus)):
            for i in range(len(sentences_words[s])):
                if cur_tags[s][i] != gold_tags[s][i]:
                    errors.append((s, i))
        if not errors:
            if verbose:
                print(f"第 {round_idx + 1} 轮：已无标注错误，学习结束。")
            break

        # 4.2 基于错误位置的真实上下文生成候选规则（保持出现顺序去重，保证结果可复现）
        candidates = []
        seen = set()
        for (s, i) in errors:
            words, tags = sentences_words[s], cur_tags[s]
            a, b = tags[i], gold_tags[s][i]
            for tname, tfunc in templates.items():
                val = tfunc(words, tags, i)
                if val is None:
                    continue
                key = (tname, a, b, val)
                if key not in seen:
                    seen.add(key)
                    candidates.append(key)

        # 4.3 评估每条候选规则在全语料上的"净修正数"
        # 若多条规则净修正数并列最高，保留先出现（更符合模板优先级）的一条
        best_rule, best_net = None, 0
        for (tname, a, b, val) in candidates:
            tfunc = templates[tname]
            good, bad = 0, 0
            for s in range(len(corpus)):
                words, tags, gt = sentences_words[s], cur_tags[s], gold_tags[s]
                for i in range(len(words)):
                    if tags[i] != a or tfunc(words, tags, i) != val:
                        continue
                    if gt[i] == b:
                        good += 1       # 修复了一个错误
                    elif gt[i] == a:
                        bad += 1        # 破坏了一个原本正确的标注
                    # 其余情况：原本就是别的错误，净错误数不变，忽略
            net = good - bad
            if net > best_net:
                best_net, best_rule = net, (tname, a, b, val, good, bad)

        if best_rule is None:
            if verbose:
                print(f"第 {round_idx + 1} 轮：没有能带来净改进的候选规则，学习结束。")
            break

        tname, a, b, val, good, bad = best_rule
        learned_rules.append({"template": tname, "from": a, "to": b, "value": val,
                               "good": good, "bad": bad, "net": best_net})
        if verbose:
            print(f"第 {round_idx + 1} 轮学到规则: 若词性={a} 且 {tname}={val}，则改为 {b}"
                  f"   (修复 {good} 处，引入新错误 {bad} 处，净改进 +{best_net})")

        # 4.4 将规则应用到整个语料，更新当前标注（基于本轮开始时的快照，避免顺序副作用）
        for s in range(len(corpus)):
            words, tags = sentences_words[s], cur_tags[s]
            to_change = [i for i in range(len(words))
                         if tags[i] == a and templates[tname](words, tags, i) == val]
            for i in to_change:
                tags[i] = b

    return learned_rules, cur_tags, gold_tags, sentences_words


learned_rules, cur_tags, gold_tags, sentences_words = learn_rules(train_corpus, lexicon)

print()
print("学到的规则列表（按学习顺序，即应用顺序）：")
for idx, r in enumerate(learned_rules, 1):
    print(f"  规则{idx}: 若词性={r['from']} 且 {r['template']}={r['value']}，则改为 {r['to']}")


第 1 轮学到规则: 若词性=NN 且 PREV-TAG=TO，则改为 VB   (修复 2 处，引入新错误 0 处，净改进 +2)
第 2 轮学到规则: 若词性=NN 且 PREV-TAG=MD，则改为 VB   (修复 1 处，引入新错误 0 处，净改进 +1)
第 3 轮：已无标注错误，学习结束。

学到的规则列表（按学习顺序，即应用顺序）：
  规则1: 若词性=NN 且 PREV-TAG=TO，则改为 VB
  规则2: 若词性=NN 且 PREV-TAG=MD，则改为 VB


学到的第一条规则 **"若词性=NN 且 PREV-TAG=TO，则改为 VB"**，
正是书中给出的经典补丁规则 **"NN VB PREV-TAG TO"**！

第二条规则 **"若词性=NN 且 PREV-TAG=MD，则改为 VB"** 则进一步修正了 "She will book the room" 这类
由情态动词引导的动词用法，体现了错误驱动学习会依次挑选"当前收益最大"的规则，直至语料中的错误被逐步消除。


## 5. 应用规则前后的标注准确率对比

In [6]:
def accuracy(tags_pred, tags_gold):
    total, correct = 0, 0
    for sp, sg in zip(tags_pred, tags_gold):
        for tp, tg in zip(sp, sg):
            total += 1
            correct += (tp == tg)
    return correct / total


baseline_tags = [[initial_tag_word(w, lexicon) for w in words] for words in sentences_words]
print(f"应用规则前（仅词典初始化）准确率: {accuracy(baseline_tags, gold_tags):.2%}")
print(f"应用全部学到规则后准确率:         {accuracy(cur_tags, gold_tags):.2%}")


应用规则前（仅词典初始化）准确率: 92.50%
应用全部学到规则后准确率:         100.00%


## 6. 复现书中例子: "to book a hotel"

用学到的规则表，对一个新句子（"to book a hotel"，其中 *hotel* 是训练语料中从未出现过的未登录词）
进行标注，观察 *book* 如何从默认错误的 NN 被逐步修正为正确的 VB。


In [7]:
def apply_learned_rules(words, lexicon, rules):
    """依次应用学到的规则表，记录每一步的标注变化"""
    tags = [initial_tag_word(w, lexicon) for w in words]
    steps = [("初始化（词典 + 默认规则）", list(tags))]
    for r in rules:
        tname, a, b, val = r["template"], r["from"], r["to"], r["value"]
        tfunc = templates[tname]
        to_change = [i for i in range(len(words)) if tags[i] == a and tfunc(words, tags, i) == val]
        if to_change:
            for i in to_change:
                tags[i] = b
            steps.append((f"应用规则 [词性={a} 且 {tname}={val} -> {b}]", list(tags)))
    return tags, steps


test_words = ["to", "book", "a", "hotel", "."]
final_tags, steps = apply_learned_rules(test_words, lexicon, learned_rules)

print("=== 复现书中例子: 'to book a hotel' ===\n")
for desc, tags in steps:
    pairs = " ".join(f"{w}/{t}" for w, t in zip(test_words, tags))
    print(f"{desc}\n  -> {pairs}\n")

print("最终标注结果:", list(zip(test_words, final_tags)))


=== 复现书中例子: 'to book a hotel' ===

初始化（词典 + 默认规则）
  -> to/TO book/NN a/DT hotel/NN ./.

应用规则 [词性=NN 且 PREV-TAG=TO -> VB]
  -> to/TO book/VB a/DT hotel/NN ./.

最终标注结果: [('to', 'TO'), ('book', 'VB'), ('a', 'DT'), ('hotel', 'NN'), ('.', '.')]


可以看到：

1. 初始化阶段，*book* 因词典中 NN 更常见而被误标为 **NN**；*hotel* 是未登录词，按默认规则也被赋予 **NN**（合理，因为它确实是名词）。
2. 应用学到的第一条规则（PREV-TAG=TO → VB）后，*book* 因为前一个词 *to* 被标注为 TO，成功被修正为正确的 **VB**。

这正是原文所描述的场景："可以用于解决类似 'to book a hotel' 中对于单词 book 的词性默认标注错误的问题。"


## 7. 方法局限性小结

基于错误驱动的规则方法可以在一定程度上缓解人工规则抽取的时间成本和人力成本，在小规模场景下也能取得不错的效果。
但正如原文指出：

- 效果**严重依赖训练语料的规模和质量**——若某种上下文模式在训练语料中从未出现，就无法学到对应规则；
- **较难处理未登录词**——本案例中的 *hotel* 恰好靠"默认规则"蒙对了词性，但更复杂的未登录词（如新造词、专业术语）未必能被正确处理；
- 受限于**规则模板的复杂度**，其效果通常也低于基于统计机器学习的方法（如 2.4.2 节的隐马尔可夫模型、后续章节的条件随机场等）。
